In [ ]:
!pip install semopy pandas

In [1]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)
n = 300 

# Create the 2x2 Factorial design (Origin: 0=No, 1=Uji; Method: 0=No, 1=Yes)
data = {
    'Origin': np.random.choice([0, 1], n),
    'Method': np.random.choice([0, 1], n)
}

# Create latent variables (hidden scores) 
# We assume Origin and Method have a positive impact on Authenticity
# We assume Authenticity has a positive impact on BuyIntention
auth_latent = 0.5 * data['Origin'] + 0.5 * data['Method'] + np.random.normal(0, 0.5, n)
buy_latent = 0.7 * auth_latent + np.random.normal(0, 0.5, n)

# Generate observed items (scaled 1-7)
# Each item is a function of the latent variable + random noise
df = pd.DataFrame(data)
df['Item1'] = np.clip(np.round(4 + 1.2 * auth_latent + np.random.normal(0, 0.5, n)), 1, 7)
df['Item2'] = np.clip(np.round(4 + 1.2 * auth_latent + np.random.normal(0, 0.5, n)), 1, 7)
df['Item3'] = np.clip(np.round(4 + 1.2 * auth_latent + np.random.normal(0, 0.5, n)), 1, 7)

df['Item4'] = np.clip(np.round(4 + 1.2 * buy_latent + np.random.normal(0, 0.5, n)), 1, 7)
df['Item5'] = np.clip(np.round(4 + 1.2 * buy_latent + np.random.normal(0, 0.5, n)), 1, 7)
df['Item6'] = np.clip(np.round(4 + 1.2 * buy_latent + np.random.normal(0, 0.5, n)), 1, 7)

# Preview the data
print(df.head())

   Origin  Method  Item1  Item2  Item3  Item4  Item5  Item6
0       0       0    4.0    5.0    4.0    4.0    4.0    3.0
1       1       0    4.0    4.0    4.0    3.0    4.0    3.0
2       0       0    5.0    5.0    5.0    5.0    5.0    5.0
3       0       1    5.0    5.0    5.0    4.0    6.0    5.0
4       0       1    4.0    4.0    3.0    3.0    3.0    3.0


In [4]:
!pip install semopy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 3.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.3/94.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 8.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 14.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 11.0 MB/s eta 0:00:000:0100:01
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import pandas as pd
import semopy

# 1. Load your dataset (assuming it is in a CSV file)
# df = pd.read_csv('your_data.csv')

# 2. Define the Model (using the lavaan-like syntax)
desc = """
    # Measurement Model (Factor Loadings)
    Authenticity =~ Item1 + Item2 + Item3
    BuyIntention =~ Item4 + Item5 + Item6
    
    # Structural Model (Regressions)
    Authenticity ~ Origin + Method
    BuyIntention ~ Authenticity
"""

# 3. Initialize and fit the model
model = semopy.Model(desc)
model.fit(df)

# 4. Get the results
stats = semopy.calc_stats(model)
print(stats.T) # Prints model fit indices (CFI, RMSEA, etc.)

# 5. View the parameter estimates (Path coefficients)
res = model.inspect()
print(res)

                    Value
DoF             21.000000
DoF Baseline    30.000000
chi2            10.591545
chi2 p-value     0.970163
chi2 Baseline  829.561978
CFI              1.013018
GFI              0.987232
AGFI             0.981761
NFI              0.987232
TLI              1.018597
RMSEA            0.000000
AIC             29.929390
BIC             85.486127
LogLik           0.035305
            lval  op          rval  Estimate  Std. Err    z-value p-value
0   Authenticity   ~        Origin  0.511673  0.075965   6.735639     0.0
1   Authenticity   ~        Method  0.440717  0.075073   5.870544     0.0
2   BuyIntention   ~  Authenticity  0.775164  0.092717   8.360543     0.0
3          Item1   ~  Authenticity  1.000000         -          -       -
4          Item2   ~  Authenticity  1.048072  0.092783  11.295929     0.0
5          Item3   ~  Authenticity  1.020415   0.08989  11.351839     0.0
6          Item4   ~  BuyIntention  1.000000         -          -       -
7          Item5  

In [6]:
# Just the measurement part (The CFA)
measurement_model = """
    Authenticity =~ Item1 + Item2 + Item3
    BuyIntention =~ Item4 + Item5 + Item6
"""
# Run as a CFA
model = semopy.Model(measurement_model)
model.fit(df)
print(model.inspect())

            lval  op          rval  Estimate  Std. Err    z-value p-value
0          Item1   ~  Authenticity  1.000000         -          -       -
1          Item2   ~  Authenticity  1.004268  0.093202  10.775124     0.0
2          Item3   ~  Authenticity  1.024969  0.092163  11.121254     0.0
3          Item4   ~  BuyIntention  1.000000         -          -       -
4          Item5   ~  BuyIntention  0.993768  0.069312  14.337707     0.0
5          Item6   ~  BuyIntention  0.982747  0.069965  14.046188     0.0
6   Authenticity  ~~  Authenticity  0.408262  0.059003    6.91932     0.0
7   Authenticity  ~~  BuyIntention  0.313863  0.044855   6.997317     0.0
8   BuyIntention  ~~  BuyIntention  0.630587  0.080051    7.87731     0.0
9          Item1  ~~         Item1  0.291694  0.036737   7.940086     0.0
10         Item2  ~~         Item2  0.392645  0.043397   9.047817     0.0
11         Item3  ~~         Item3  0.323212  0.039614   8.159138     0.0
12         Item4  ~~         Item4  0.

In [8]:
df

,Origin,Method,Item1,Item2,Item3,Item4,Item5,Item6
0,0,0,4.0,5.0,4.0,4.0,4.0,3.0
1,1,0,4.0,4.0,4.0,3.0,4.0,3.0
2,0,0,5.0,5.0,5.0,5.0,5.0,5.0
3,0,1,5.0,5.0,5.0,4.0,6.0,5.0
4,0,1,4.0,4.0,3.0,3.0,3.0,3.0
...,...,...,...,...,...,...,...,...
295,1,1,5.0,5.0,6.0,5.0,4.0,5.0
296,1,0,5.0,5.0,5.0,4.0,5.0,4.0
297,0,0,4.0,3.0,4.0,3.0,4.0,3.0
298,0,1,5.0,5.0,5.0,4.0,4.0,4.0


In [9]:
import pandas as pd
import semopy

# 1. Load your data
# df = pd.read_csv('your_data.csv')

# 2. Define the Mediated Model (The theory we hypothesized)
mediated_model_desc = """
    # Measurement
    Authenticity =~ Item1 + Item2 + Item3
    BuyIntention =~ Item4 + Item5 + Item6
    
    # Structural
    Authenticity ~ Origin + Method
    BuyIntention ~ Authenticity
"""

# 3. Define the Direct Model (The rival theory)
# Here, we treat BuyIntention as a simple observed variable 
# to compare against the latent model.
# The Direct Model
direct_model_desc = """
    # Define the factor
    BuyIntention =~ Item4 + Item5 + Item6
    
    # Regress the factor directly on the IVs
    BuyIntention ~ Origin + Method
"""

# 4. Fit both models
model_a = semopy.Model(mediated_model_desc)
model_a.fit(df)

model_b = semopy.Model(direct_model_desc)
model_b.fit(df)

# 5. Get statistics for comparison
stats_a = semopy.calc_stats(model_a)
stats_b = semopy.calc_stats(model_b)

print("--- Mediated Model Stats ---")
print(stats_a[['AIC', 'BIC', 'CFI', 'RMSEA']].T)

print("\n--- Direct Model Stats ---")
print(stats_b[['AIC', 'BIC']].T)

--- Mediated Model Stats ---
           Value
AIC    29.929390
BIC    85.486127
CFI     1.013018
RMSEA   0.000000

--- Direct Model Stats ---
         Value
AIC  15.987221
BIC  45.617480


In [10]:
!pip install factor_analyzer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for factor_analyzer: filename=factor_analyzer-0.5.1-py2.py3-none-any.whl size=42655 sha256=9026b8be6432a7138d8b16311916a7049e5eb50ad46ecfbaef46245c62f77c2b
  Stored in directory: /Users/pachara_win/Library/Caches/pip/wheels/24/59/82/6493618e30ed1cb7a013b9e1b0c9e17de80b04dfcef4ba8a4d
Successfully built factor_analyzer


In [11]:
import pandas as pd
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

# 1. Load your data (assuming df is your dataframe of items)
# df = df[['Item1', 'Item2', 'Item3', 'Item4', 'Item5', 'Item6']]

# 2. Check if Factor Analysis is appropriate
# Bartlett’s test should be < 0.05
chi_square_value, p_value = calculate_bartlett_sphericity(df)
print(f"Bartlett's Test p-value: {p_value}")

# KMO test should be > 0.6
kmo_all, kmo_model = calculate_kmo(df)
print(f"KMO Score: {kmo_model}")

# 3. Perform Factor Analysis
# We set n_factors=2 because we hypothesize 2 factors: Authenticity and Buying Intention
fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(df)

# 4. View the Factor Loadings
loadings = pd.DataFrame(fa.loadings_, index=df.columns, columns=['Factor1', 'Factor2'])
print("\nFactor Loadings Table:")
print(loadings)

Bartlett's Test p-value: 5.3827735359028495e-154
KMO Score: 0.8359112500416966

Factor Loadings Table:
         Factor1   Factor2
Origin  0.174080  0.321852
Method  0.067988  0.312403
Item1   0.247046  0.719422
Item2   0.262905  0.700876
Item3   0.260797  0.695392
Item4   0.772358  0.240227
Item5   0.791937  0.239376
Item6   0.748086  0.275016


/opt/miniconda3/envs/capstone/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [12]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Calculate a combined score for Authenticity (the average of your 3 items)
df['Authenticity_Score'] = df[['Item1', 'Item2', 'Item3']].mean(axis=1)

# Run a Two-Way ANOVA
# We want to see how Origin and Method affect the Authenticity_Score
model = ols('Authenticity_Score ~ C(Origin) * C(Method)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print(anova_table)

                         sum_sq     df          F        PR(>F)
C(Origin)             20.341748    1.0  49.049169  1.682473e-11
C(Method)             16.037289    1.0  38.670015  1.705747e-09
C(Origin):C(Method)    0.115063    1.0   0.277447  5.987738e-01
Residual             122.757584  296.0        NaN           NaN
